# Qualtrics VOC Sentiment Analysis — End-to-End Walkthrough

This notebook demonstrates the full Voice of Customer (VOC) analytics pipeline:
1. Simulate Qualtrics API data export
2. Preprocess open-text survey responses
3. VADER sentiment scoring
4. TF-IDF theme extraction
5. OKR health monitoring
6. Executive visualisations


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

from qualtrics_export import export_survey_responses, check_okr_thresholds
from sentiment_analysis import score_responses, generate_voc_report, extract_themes

plt.rcParams['figure.facecolor'] = 'white'
sns.set_style('whitegrid')
print('All modules loaded.')

## Step 1 — Qualtrics API Export (OAuth 2.0 simulation)

In [ ]:
responses = export_survey_responses()
okr_check = check_okr_thresholds(responses)
print(f'Exported {len(responses)} responses')
print(f'OKR Status: {okr_check["status"]}')
print(f'NPS: {okr_check["nps"]}  |  Avg CSAT: {okr_check["avg_csat"]}')

## Step 2 — Load and Preview Data

In [ ]:
df = pd.read_csv('data/sample_survey_responses.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print('\nData types:')
print(df.dtypes)

## Step 3 — NPS and CSAT Baseline

In [ ]:
nps_scores = df['nps_score'].astype(int)
promoters  = (nps_scores >= 9).sum()
passives   = ((nps_scores >= 7) & (nps_scores <= 8)).sum()
detractors = (nps_scores <= 6).sum()
nps        = round(((promoters - detractors) / len(nps_scores)) * 100, 1)
avg_csat   = round(df['csat_score'].mean(), 2)

print(f'Net Promoter Score (NPS): {nps}')
print(f'  Promoters : {promoters}  ({round(promoters/len(nps_scores)*100,1)}%)')
print(f'  Passives  : {passives}  ({round(passives/len(nps_scores)*100,1)}%)')
print(f'  Detractors: {detractors}  ({round(detractors/len(nps_scores)*100,1)}%)')
print(f'\nAvg CSAT: {avg_csat} / 5.0')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# NPS distribution
axes[0].hist(df['nps_score'], bins=11, range=(0,10), color='#185FA5', edgecolor='white')
axes[0].set_xlabel('NPS Score')
axes[0].set_ylabel('Count')
axes[0].set_title(f'NPS Score Distribution  (NPS = {nps})')
axes[0].axvline(6.5, color='orange', linestyle='--', label='Detractor/Passive boundary')
axes[0].axvline(8.5, color='green',  linestyle='--', label='Passive/Promoter boundary')
axes[0].legend(fontsize=8)

# CSAT distribution
axes[1].hist(df['csat_score'], bins=5, range=(1,6), color='#1D9E75', edgecolor='white')
axes[1].set_xlabel('CSAT Score')
axes[1].set_ylabel('Count')
axes[1].set_title(f'CSAT Distribution  (Avg = {avg_csat})')
axes[1].axvline(avg_csat, color='red', linestyle='--', label=f'Mean = {avg_csat}')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('data/nps_csat_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4 — VADER Sentiment Scoring

In [ ]:
df_scored = score_responses(df)
print('Sentiment distribution:')
print(df_scored['sentiment'].value_counts())
print('\nSample scored responses:')
df_scored[['response_id', 'open_text', 'vader_compound', 'sentiment']].head(8)

In [ ]:
COLORS = {'Positive': '#1D9E75', 'Neutral': '#888780', 'Negative': '#D85A30'}

fig, ax = plt.subplots(figsize=(8, 4))
for sentiment, grp in df_scored.groupby('sentiment'):
    ax.hist(grp['vader_compound'], bins=12, alpha=0.65,
            color=COLORS[sentiment], label=sentiment, edgecolor='white')
ax.axvline( 0.05, color='green', linestyle='--', linewidth=1, label='Pos threshold')
ax.axvline(-0.05, color='red',   linestyle='--', linewidth=1, label='Neg threshold')
ax.set_xlabel('VADER Compound Score')
ax.set_ylabel('Count')
ax.set_title('VADER Compound Score Distribution by Sentiment Class')
ax.legend()
plt.tight_layout()
plt.savefig('data/vader_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5 — TF-IDF Theme Extraction

In [ ]:
for label in ['Positive', 'Neutral', 'Negative']:
    texts  = df_scored.loc[df_scored['sentiment'] == label, 'cleaned_text'].tolist()
    themes = extract_themes(texts, top_n=6)
    print(f'{label:10s} themes: {themes}')

## Step 6 — Customer Journey Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sentiment by channel
channel_df = (
    df_scored.groupby(['channel', 'sentiment'])
    .size().unstack(fill_value=0)
    .reindex(columns=['Positive', 'Neutral', 'Negative'], fill_value=0)
)
channel_df.plot(kind='bar', ax=axes[0],
                color=[COLORS['Positive'], COLORS['Neutral'], COLORS['Negative']],
                edgecolor='white', rot=0)
axes[0].set_title('Sentiment by Channel')
axes[0].set_ylabel('Responses')
axes[0].legend(title='Sentiment')

# Avg CSAT by product surface
surface_csat = df_scored.groupby('product_surface')['csat_score'].mean().sort_values()
colors = ['#D85A30' if v < 3.8 else '#1D9E75' for v in surface_csat.values]
surface_csat.plot(kind='barh', ax=axes[1], color=colors, edgecolor='white')
axes[1].axvline(3.8, color='orange', linestyle='--', label='CSAT Target (3.8)')
axes[1].set_title('Avg CSAT by Product Surface')
axes[1].set_xlabel('Avg CSAT Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('data/customer_journey_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7 — OKR Health Report

In [ ]:
report = generate_voc_report(df_scored)
print(f"OKR Status   : {report['okr_status']}")
print(f"NPS Score    : {report['overall_stats']['nps_score']}  (target: 30)")
print(f"Avg CSAT     : {report['overall_stats']['avg_csat']}  (target: 3.8)")
if report['okr_alerts']:
    print('\nActive alerts:')
    for a in report['okr_alerts']:
        print(f'  ⚠  {a["message"]}')
else:
    print('\nNo OKR threshold breaches detected.')

## Step 8 — Low CSAT Verbatim Review

In [ ]:
low_csat_df = df_scored[df_scored['csat_score'].astype(float) <= 2].sort_values('vader_compound')
print(f'Low CSAT responses (score ≤ 2): {len(low_csat_df)}\n')
for _, row in low_csat_df.head(5).iterrows():
    print(f"[{row['sentiment']:8s} | CSAT {row['csat_score']} | NPS {row['nps_score']}]  {row['open_text']}")
    print()